# Comparação — BRICS AI Governance Declaration × OECD Recommendation on AI

Este notebook compara os dois documentos já extraídos (`BRICS.json` e
`OCDE.json`) em dois eixos:

1. **Similaridade textual** — o quanto o vocabulário e a ênfase temática dos
   dois documentos se sobrepõem, usando o índice de Jaccard (sobreposição de
   vocabulário) e a similaridade de cosseno (sobreposição ponderada por
   frequência dos termos);
2. **Divergência/convergência por termo** — quais termos cada documento
   prioriza mais (gráfico *tornado*) e como os termos compartilhados se
   distribuem entre os dois documentos (dispersão BRICS × OCDE).

## Importação, carregamento e tokenização

In [ ]:
import json
import math
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

STOPWORDS = {
    "the", "a", "an", "and", "of", "to", "in", "that", "for", "as", "on",
    "with", "by", "or", "be", "is", "are", "we", "our", "their", "all",
    "at", "from", "this", "these", "those", "such", "which", "it", "its",
    "within", "while", "also", "both", "other", "than", "then", "so",
    "but", "not", "no", "do", "does", "did", "has", "have", "had", "was",
    "were", "been", "being", "will", "would", "should", "could", "can",
    "may", "might", "must", "shall", "us", "any", "into", "across", "over",
    "under", "about", "between", "including", "particularly", "especially",
    "towards", "toward", "through", "among", "who", "what", "how", "if",
    "each", "more", "most", "some", "own", "they", "them", "there",
}

def load_tokens(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    text = " ".join(section["text"] for section in data["sections"])
    tokens = re.findall(r"[a-zA-Z]+(?:-[a-zA-Z]+)*", text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2], data

BRICS_LABEL = "BRICS Declaration"
OECD_LABEL = "OECD Recommendation"

brics_tokens, brics_data = load_tokens("BRICS.json")
ocde_tokens, ocde_data = load_tokens("OCDE.json")

brics_counts = Counter(brics_tokens)
ocde_counts = Counter(ocde_tokens)

print(f"{BRICS_LABEL}: {len(brics_tokens)} tokens, {len(brics_counts)} termos únicos")
print(f"{OECD_LABEL}: {len(ocde_tokens)} tokens, {len(ocde_counts)} termos únicos")


## 1. Similaridade entre os textos

Duas métricas complementares:

- **Índice de Jaccard** — proporção de termos que aparecem nos *dois*
  documentos em relação ao total de termos distintos usados por qualquer um
  deles. Mede sobreposição pura de vocabulário, sem considerar frequência.
- **Similaridade de cosseno** — trata cada documento como um vetor de
  frequências relativas (por 1.000 tokens) sobre o vocabulário combinado.
  Mede o quanto os dois documentos "apontam na mesma direção" temática,
  dando mais peso a termos usados com frequência semelhante nos dois.

In [ ]:
def relative_freq(counter, total_tokens, term):
    return counter.get(term, 0) / total_tokens * 1000

vocab_brics = set(brics_counts)
vocab_ocde = set(ocde_counts)
vocab_union = vocab_brics | vocab_ocde
vocab_intersection = vocab_brics & vocab_ocde

jaccard = len(vocab_intersection) / len(vocab_union)

terms = sorted(vocab_union)
vec_brics = [relative_freq(brics_counts, len(brics_tokens), t) for t in terms]
vec_ocde = [relative_freq(ocde_counts, len(ocde_tokens), t) for t in terms]

dot = sum(a * b for a, b in zip(vec_brics, vec_ocde))
norm_brics = math.sqrt(sum(a * a for a in vec_brics))
norm_ocde = math.sqrt(sum(b * b for b in vec_ocde))
cosine_similarity = dot / (norm_brics * norm_ocde)

print(f"Vocabulário BRICS:        {len(vocab_brics)} termos")
print(f"Vocabulário OCDE:         {len(vocab_ocde)} termos")
print(f"Termos em comum:         {len(vocab_intersection)} termos")
print(f"Índice de Jaccard:       {jaccard:.3f}  ({jaccard*100:.1f}% do vocabulário combinado é compartilhado)")
print(f"Similaridade de cosseno: {cosine_similarity:.3f}  (1 = idêntico, 0 = nenhuma sobreposição de ênfase)")


### Gráfico — as duas métricas de similaridade lado a lado

Ambas as métricas ficam entre 0 e 1, o que permite compará-las diretamente
na mesma escala.

In [ ]:
SEQUENTIAL_BLUE = "#2a78d6"
INK_PRIMARY = "#0b0b0b"
INK_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
SURFACE = "#fcfcfb"

def plot_ranked_bar(pairs, title, xlabel="Frequência (nº de ocorrências)", xlim_max=None, value_fmt="{:.0f}", filename=None):
    labels = [p[0] for p in pairs][::-1]
    values = [p[1] for p in pairs][::-1]

    fig, ax = plt.subplots(figsize=(8, max(2.2, 0.45 * len(pairs))), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)

    bars = ax.barh(labels, values, color=SEQUENTIAL_BLUE, height=0.55, zorder=3)

    ax.set_xlabel(xlabel, color=INK_MUTED, fontsize=10)
    ax.set_title(title, color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

    ax.tick_params(axis="y", colors=INK_PRIMARY, labelsize=10)
    ax.tick_params(axis="x", colors=INK_MUTED, labelsize=9)

    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRIDLINE)

    ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

    max_val = xlim_max if xlim_max is not None else max(values)
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_width() + max_val * 0.015,
            bar.get_y() + bar.get_height() / 2,
            value_fmt.format(value),
            va="center", ha="left",
            color=INK_PRIMARY, fontsize=9,
        )
    ax.set_xlim(0, max_val * 1.15)

    fig.tight_layout()
    if filename:
        fig.savefig(filename, dpi=150, facecolor=SURFACE, bbox_inches="tight")
    plt.show()

plot_ranked_bar(
    [("Índice de Jaccard\n(sobreposição de vocabulário)", jaccard),
     ("Similaridade de cosseno\n(sobreposição de ênfase)", cosine_similarity)],
    "Similaridade textual — BRICS Declaration × OECD Recommendation",
    xlabel="Similaridade (0 a 1)",
    xlim_max=1.0,
    value_fmt="{:.3f}",
)


### Termos mais compartilhados

Os termos com maior frequência combinada nos dois documentos — o núcleo de
vocabulário que os dois textos têm efetivamente em comum.

In [ ]:
MIN_COMBINED_COUNT = 4

shared_terms = [
    (t, brics_counts[t] + ocde_counts[t])
    for t in vocab_intersection
    if brics_counts[t] + ocde_counts[t] >= MIN_COMBINED_COUNT
]
shared_terms.sort(key=lambda x: x[1], reverse=True)

plot_ranked_bar(
    shared_terms[:15],
    "Termos mais compartilhados entre os dois documentos (nº total de ocorrências)",
)


## 2. Divergência e convergência por termo

### Gráfico tornado — quem prioriza cada termo

Para os termos com pelo menos 4 ocorrências combinadas, comparamos a
frequência relativa (por 1.000 tokens) em cada documento. O gráfico mostra,
de um lado, os termos mais característicos do BRICS e, do outro, os mais
característicos da OCDE — ordenados pela diferença entre os dois.

In [ ]:
CATEGORICAL_BLUE = "#2a78d6"   # BRICS
CATEGORICAL_AQUA = "#1baf7a"   # OECD
TOP_N_EACH_SIDE = 12

rows = []
for t in vocab_union:
    b = relative_freq(brics_counts, len(brics_tokens), t)
    o = relative_freq(ocde_counts, len(ocde_tokens), t)
    combined_count = brics_counts.get(t, 0) + ocde_counts.get(t, 0)
    if combined_count >= MIN_COMBINED_COUNT:
        rows.append((t, b, o, b - o))

rows.sort(key=lambda x: x[3])
oecd_leaning = rows[:TOP_N_EACH_SIDE]                 # diferença mais negativa -> OCDE prioriza mais
brics_leaning = rows[-TOP_N_EACH_SIDE:]               # diferença mais positiva -> BRICS prioriza mais
tornado_rows = oecd_leaning + brics_leaning
tornado_rows.sort(key=lambda x: x[3])                 # OCDE no topo, BRICS na base

labels = [r[0] for r in tornado_rows]
brics_vals = [r[1] for r in tornado_rows]
ocde_vals = [r[2] for r in tornado_rows]

fig, ax = plt.subplots(figsize=(9, 0.4 * len(tornado_rows) + 1), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = range(len(tornado_rows))
ax.barh(y_pos, [-v for v in brics_vals], color=CATEGORICAL_BLUE, height=0.6, zorder=3, label=BRICS_LABEL)
ax.barh(y_pos, ocde_vals, color=CATEGORICAL_AQUA, height=0.6, zorder=3, label=OECD_LABEL)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9)
ax.axvline(0, color=GRIDLINE, linewidth=1.0, zorder=2)

ax.set_xlabel("Frequência relativa (ocorrências por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title(
    "Termos que cada documento prioriza — BRICS (esquerda) × OCDE (direita)",
    color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14,
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)

max_abs = max(max(brics_vals), max(ocde_vals))
ax.set_xlim(-max_abs * 1.15, max_abs * 1.15)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.0f}" for x in xticks], color=INK_MUTED, fontsize=9)

ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### Dispersão BRICS × OCDE — convergência e divergência

Cada ponto é um termo (com pelo menos 4 ocorrências combinadas), posicionado
pela frequência relativa no BRICS (eixo X) e na OCDE (eixo Y). A linha
diagonal tracejada marca "mesma prioridade nos dois documentos": pontos
próximos a ela **convergem**; pontos afastados da diagonal — grudados em um
dos eixos — **divergem**, revelando o vocabulário específico de cada
documento.

In [ ]:
CATEGORICAL_YELLOW = "#eda100"  # termos convergentes (equilíbrio entre os dois documentos)
BALANCE_RATIO_THRESHOLD = 0.6    # min(b, o) / max(b, o) >= 0.6 => convergente

scatter_rows = [r for r in rows]  # já filtrado por MIN_COMBINED_COUNT acima

def classify(b, o):
    ratio = min(b, o) / max(b, o) if max(b, o) > 0 else 1.0
    if ratio >= BALANCE_RATIO_THRESHOLD:
        return "convergente"
    return "BRICS" if b > o else "OCDE"

groups = {"convergente": [], "BRICS": [], "OCDE": []}
for t, b, o, _ in scatter_rows:
    groups[classify(b, o)].append((t, b, o))

fig, ax = plt.subplots(figsize=(8, 8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

max_val = max(max(b, o) for _, b, o, _ in scatter_rows) * 1.1
ax.plot([0, max_val], [0, max_val], linestyle="--", linewidth=1.2, color=INK_MUTED, zorder=2)

color_map = {"BRICS": CATEGORICAL_BLUE, "OCDE": CATEGORICAL_AQUA, "convergente": CATEGORICAL_YELLOW}
label_map = {"BRICS": BRICS_LABEL, "OCDE": OECD_LABEL, "convergente": "Convergente (equilíbrio)"}

for group_name in ["convergente", "BRICS", "OCDE"]:
    pts = groups[group_name]
    if not pts:
        continue
    xs = [p[1] for p in pts]
    ys = [p[2] for p in pts]
    ax.scatter(xs, ys, s=46, color=color_map[group_name], alpha=0.85, zorder=3,
               edgecolors=SURFACE, linewidths=0.6, label=label_map[group_name])

# Rotula os termos mais extremos em cada grupo para dar contexto ao gráfico
def top_extreme(pts, key, n=4):
    return sorted(pts, key=key, reverse=True)[:n]

annotate_targets = (
    top_extreme(groups["BRICS"], key=lambda p: p[1] - p[2])
    + top_extreme(groups["OCDE"], key=lambda p: p[2] - p[1])
    + top_extreme(groups["convergente"], key=lambda p: p[1] + p[2])
)
for t, x, y in annotate_targets:
    ax.annotate(t, (x, y), textcoords="offset points", xytext=(6, 4),
                fontsize=8, color=INK_PRIMARY)

ax.set_xlabel(f"Frequência relativa — {BRICS_LABEL} (por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_ylabel(f"Frequência relativa — {OECD_LABEL} (por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title("Convergência e divergência de termos — BRICS × OCDE", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

ax.set_xlim(0, max_val)
ax.set_ylim(0, max_val)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)

ax.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

ax.text(max_val * 0.02, max_val * 0.97, "↑ mais enfatizado pela OCDE", color=INK_MUTED, fontsize=9, va="top")
ax.text(max_val * 0.97, max_val * 0.03, "→ mais enfatizado pelo BRICS", color=INK_MUTED, fontsize=9, ha="right")

ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=9, bbox_to_anchor=(0.02, 0.9))

fig.tight_layout()
plt.show()


## Síntese

- A similaridade de vocabulário bruta (Jaccard) é moderada-baixa e a
  similaridade de cosseno (que pondera pela frequência) é moderada — os dois
  documentos tratam do mesmo tema (governança de IA), mas com vocabulário e
  ênfases distintos.
- O **BRICS** prioriza termos ligados a desenvolvimento, países em
  desenvolvimento, economia digital e cooperação internacional
  (*countries*, *digital*, *development*, *developing*, *economy*,
  *technologies*, *innovation*, *international*, *governance*).
- A **OCDE** prioriza termos ligados à arquitetura de governança do próprio
  sistema de IA e aos atores responsáveis por ele (*system*, *lifecycle*,
  *actors*, *stakeholders*, *governments*, *trustworthy*, *policy*,
  *recommendation*, *principles*).
- Termos como *development*, *systems*, *data*, *digital*, *rights* e
  *governance* aparecem com peso relevante nos dois textos e formam o núcleo
  comum de vocabulário entre as duas declarações.

## 3. PBIA — a qual dos dois documentos o Brasil se alinha mais?

O PBIA (`PBIA.json`) é o Plano Brasileiro de Inteligência Artificial. Ele é
tokenizado com exatamente o mesmo método usado para BRICS e OCDE (mesma lista
de *stopwords*, mesmo campo `sections`), para que as três séries de
frequência sejam diretamente comparáveis. A pergunta desta seção: o
vocabulário do PBIA se aproxima mais do eixo **desenvolvimento / países em
desenvolvimento / economia digital** do BRICS, ou do eixo **arquitetura de
governança / atores responsáveis** da OCDE?

In [ ]:
PBIA_LABEL = "PBIA (Brasil)"

pbia_tokens, pbia_data = load_tokens("PBIA.json")
pbia_counts = Counter(pbia_tokens)

print(f"{PBIA_LABEL}: {len(pbia_tokens)} tokens, {len(pbia_counts)} termos únicos")


### 3.1. Similaridade do PBIA com cada documento

Reaproveitamos o índice de Jaccard e a similaridade de cosseno definidos na
Seção 1, agora aplicados aos pares (PBIA, BRICS) e (PBIA, OCDE).

In [ ]:
def jaccard_cosine(counts_a, tokens_a, counts_b, tokens_b):
    vocab_a = set(counts_a)
    vocab_b = set(counts_b)
    union = vocab_a | vocab_b
    inter = vocab_a & vocab_b
    jac = len(inter) / len(union)

    terms_u = sorted(union)
    va = [relative_freq(counts_a, len(tokens_a), t) for t in terms_u]
    vb = [relative_freq(counts_b, len(tokens_b), t) for t in terms_u]
    dot = sum(x * y for x, y in zip(va, vb))
    norm_a = math.sqrt(sum(x * x for x in va))
    norm_b = math.sqrt(sum(y * y for y in vb))
    cos = dot / (norm_a * norm_b) if norm_a and norm_b else 0.0
    return jac, cos

jaccard_pbia_brics, cosine_pbia_brics = jaccard_cosine(pbia_counts, pbia_tokens, brics_counts, brics_tokens)
jaccard_pbia_ocde, cosine_pbia_ocde = jaccard_cosine(pbia_counts, pbia_tokens, ocde_counts, ocde_tokens)

print(f"PBIA × {BRICS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_pbia_brics:.3f}")
print(f"  Similaridade de cosseno: {cosine_pbia_brics:.3f}")
print()
print(f"PBIA × {OECD_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_pbia_ocde:.3f}")
print(f"  Similaridade de cosseno: {cosine_pbia_ocde:.3f}")
print()
print(f"(referência) {BRICS_LABEL} × {OECD_LABEL}: Jaccard={jaccard:.3f}  Cosseno={cosine_similarity:.3f}")


### Gráfico — com qual documento o PBIA se parece mais?

As mesmas duas métricas, lado a lado, para os dois pares de comparação.

In [ ]:
CATEGORICAL_PBIA = "#e2574c"  # cor de identidade do PBIA nos gráficos desta seção

metric_labels = ["Índice de Jaccard\n(vocabulário)", "Similaridade de cosseno\n(ênfase)"]
brics_values = [jaccard_pbia_brics, cosine_pbia_brics]
ocde_values = [jaccard_pbia_ocde, cosine_pbia_ocde]

fig, ax = plt.subplots(figsize=(8, 5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = list(range(len(metric_labels)))
bar_h = 0.32

ax.barh([y + bar_h / 2 + 0.02 for y in y_pos], brics_values, height=bar_h,
        color=CATEGORICAL_BLUE, zorder=3, label=f"PBIA × {BRICS_LABEL}")
ax.barh([y - bar_h / 2 - 0.02 for y in y_pos], ocde_values, height=bar_h,
        color=CATEGORICAL_AQUA, zorder=3, label=f"PBIA × {OECD_LABEL}")

for y, v in zip(y_pos, brics_values):
    ax.text(v + 0.012, y + bar_h / 2 + 0.02, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=9)
for y, v in zip(y_pos, ocde_values):
    ax.text(v + 0.012, y - bar_h / 2 - 0.02, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(metric_labels, color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Similaridade (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_xlim(0, max(brics_values + ocde_values) * 1.3)
ax.set_title("Com qual documento o PBIA se parece mais?", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 3.2. O vocabulário mais usado pelo PBIA ecoa mais no BRICS ou na OCDE?

Para os termos mais frequentes do PBIA (excluindo palavras autorreferentes
como *brazil*, *brazilian* e *pbia*), comparamos o quanto **BRICS** e
**OCDE** usam cada um desses mesmos termos. Isso revela se o vocabulário que
o Brasil prioriza no seu próprio plano é mais eco do discurso do BRICS ou da
OCDE.

In [ ]:
SELF_REFERENTIAL = {"pbia", "brazil", "brazilian"}
MIN_PBIA_COUNT = 3
TOP_N_PBIA_TERMS = 18

pbia_terms_rows = []
for t, pbia_c in pbia_counts.most_common():
    if t in SELF_REFERENTIAL or pbia_c < MIN_PBIA_COUNT:
        continue
    b = relative_freq(brics_counts, len(brics_tokens), t)
    o = relative_freq(ocde_counts, len(ocde_tokens), t)
    pbia_terms_rows.append((t, b, o, b - o))
    if len(pbia_terms_rows) >= TOP_N_PBIA_TERMS:
        break

pbia_terms_rows.sort(key=lambda x: x[3])  # OCDE-leaning no topo, BRICS-leaning na base

labels = [r[0] for r in pbia_terms_rows]
brics_vals = [r[1] for r in pbia_terms_rows]
ocde_vals = [r[2] for r in pbia_terms_rows]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(pbia_terms_rows) + 1), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = range(len(pbia_terms_rows))
ax.barh(y_pos, [-v for v in brics_vals], color=CATEGORICAL_BLUE, height=0.6, zorder=3, label=BRICS_LABEL)
ax.barh(y_pos, ocde_vals, color=CATEGORICAL_AQUA, height=0.6, zorder=3, label=OECD_LABEL)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9)
ax.axvline(0, color=GRIDLINE, linewidth=1.0, zorder=2)

ax.set_xlabel("Frequência relativa em cada documento (ocorrências por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Vocabulário mais usado pelo PBIA: quem mais o ecoa — {BRICS_LABEL} (esquerda) × {OECD_LABEL} (direita)",
    color=INK_PRIMARY, fontsize=12, fontweight="bold", pad=14,
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)

max_abs = max(max(brics_vals), max(ocde_vals), 1e-6)
ax.set_xlim(-max_abs * 1.2, max_abs * 1.2)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.0f}" for x in xticks], color=INK_MUTED, fontsize=9)

ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 3.3. Mapa de alinhamento — PBIA entre BRICS e OCDE

Eixo X: similaridade do PBIA com o **BRICS Declaration**. Eixo Y:
similaridade do PBIA com a **OECD Recommendation**. O PBIA aparece como uma
bola (uma para cada métrica — cosseno, o ponto principal, e Jaccard, o ponto
menor e mais claro) posicionada pelas similaridades calculadas na Seção 3.1:
quanto mais **à direita** (eixo X), mais alinhado ao BRICS; quanto mais
**para cima** (eixo Y), mais alinhado à OCDE. Como referência, marcamos
também onde o próprio BRICS e a própria OCDE cairiam neste mapa (cada
documento tem similaridade 1 consigo mesmo e a similaridade cruzada
BRICS×OCDE calculada na Seção 1 com o outro).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

axis_max = 1.05
ax.plot([0, axis_max], [0, axis_max], linestyle="--", linewidth=1.2, color=INK_MUTED, zorder=2)

# Pontos de referência: onde o próprio BRICS e a própria OCDE cairiam neste mapa
ax.scatter([1.0], [cosine_similarity], s=140, color=CATEGORICAL_BLUE, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{BRICS_LABEL} (referência)")
ax.annotate(BRICS_LABEL, (1.0, cosine_similarity), textcoords="offset points", xytext=(-8, 8),
            ha="right", fontsize=8.5, color=INK_MUTED)

ax.scatter([cosine_similarity], [1.0], s=140, color=CATEGORICAL_AQUA, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{OECD_LABEL} (referência)")
ax.annotate(OECD_LABEL, (cosine_similarity, 1.0), textcoords="offset points", xytext=(8, -10),
            ha="left", fontsize=8.5, color=INK_MUTED)

# PBIA — ponto principal (similaridade de cosseno) e ponto secundário (Jaccard), ligados por uma linha fina
ax.plot([jaccard_pbia_brics, cosine_pbia_brics], [jaccard_pbia_ocde, cosine_pbia_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_PBIA, zorder=4)

ax.scatter([jaccard_pbia_brics], [jaccard_pbia_ocde], s=110, color=CATEGORICAL_PBIA, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="PBIA (Jaccard)")
ax.scatter([cosine_pbia_brics], [cosine_pbia_ocde], s=340, color=CATEGORICAL_PBIA, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="PBIA (cosseno)")
ax.annotate("PBIA", (cosine_pbia_brics, cosine_pbia_ocde), textcoords="offset points", xytext=(10, 10),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

ax.set_xlabel(f"Similaridade com o {BRICS_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_ylabel(f"Similaridade com a {OECD_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_title("PBIA no mapa BRICS × OCDE — a qual documento o Brasil se aproxima mais?",
             color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

ax.set_xlim(0, axis_max)
ax.set_ylim(0, axis_max)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

ax.text(0.02, axis_max * 0.98, "↑ mais alinhado à OCDE", color=INK_MUTED, fontsize=9, va="top")
ax.text(axis_max * 0.98, 0.02, "→ mais alinhado ao BRICS", color=INK_MUTED, fontsize=9, ha="right")

ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5, bbox_to_anchor=(0.02, 0.86))

fig.tight_layout()
plt.show()


## Síntese — PBIA, BRICS e OCDE

- Nas duas métricas, o **PBIA se aproxima mais do BRICS do que da OCDE**:
  similaridade de cosseno de **0,54** com o BRICS contra **0,40** com a OCDE,
  e Jaccard de **0,22** contra **0,20**. A diferença é mais nítida na
  similaridade de cosseno (que pondera pela frequência de uso), sugerindo
  que a ênfase temática do PBIA — não apenas o vocabulário bruto — segue
  mais de perto o BRICS.
- O vocabulário mais usado pelo PBIA (*development*, *artificial*,
  *intelligence*, *technological*, *national*, *data*, *innovation*,
  *technologies*, *country*, *global*) ecoa fortemente no BRICS, que
  compartilha a mesma ênfase em desenvolvimento tecnológico nacional,
  inovação e cooperação entre países em desenvolvimento.
- Ainda assim, o PBIA carrega um traço de vocabulário mais próximo da OCDE
  em termos como *responsible*, *plan*, *economic* e *use* — reflexo direto
  do Eixo 5 do Plano ("Support for the Regulatory and Governance Process of
  AI") e dos princípios de *Responsible AI* descritos no documento, que
  dialogam com a linguagem de governança e responsabilidade típica da OCDE.
- No mapa BRICS × OCDE, a bola do PBIA fica claramente **puxada para o eixo
  X** (mais perto do BRICS do que da OCDE), mas ainda distante do canto
  onde o próprio BRICS estaria — evidência de que o PBIA tem um vocabulário
  próprio e mais amplo (ações, planos, setores, investimento), não sendo uma
  cópia do discurso de nenhum dos dois blocos, mas com uma inclinação clara
  para a agenda de desenvolvimento tecnológico nacional que caracteriza o
  BRICS.

## 4. AI+ (China) — a qual documento a política chinesa se aproxima mais?

O AI+ (`AI_PLUS.json`) é a política *Opinions of the State Council on
Deepening the Implementation of the "Artificial Intelligence+" Initiative*,
do Conselho de Estado da China. Ele é tokenizado com exatamente o mesmo
método usado para BRICS, OCDE e PBIA (mesma lista de *stopwords*, mesmo
campo `sections`, mesmo filtro de tamanho mínimo de palavra), para que todas
as séries de frequência sejam diretamente comparáveis.

Comparamos o AI+ com os três documentos já carregados — BRICS Declaration,
OECD Recommendation e PBIA — para responder: o vocabulário e a ênfase
temática do AI+ se aproximam mais do eixo **desenvolvimento / países em
desenvolvimento / economia digital** (BRICS), do eixo **arquitetura de
governança / atores responsáveis** (OCDE), ou do plano nacional brasileiro
(PBIA)?

In [ ]:
AIPLUS_LABEL = "AI+ (China)"

aiplus_tokens, aiplus_data = load_tokens("AI_PLUS.json")
aiplus_counts = Counter(aiplus_tokens)

print(f"{AIPLUS_LABEL}: {len(aiplus_tokens)} tokens, {len(aiplus_counts)} termos únicos")


### 4.1. Similaridade do AI+ com cada documento

Reaproveitamos o índice de Jaccard e a similaridade de cosseno definidos na
Seção 1, agora aplicados aos pares (AI+, BRICS), (AI+, OCDE) e (AI+, PBIA).

In [ ]:
jaccard_aiplus_brics, cosine_aiplus_brics = jaccard_cosine(aiplus_counts, aiplus_tokens, brics_counts, brics_tokens)
jaccard_aiplus_ocde, cosine_aiplus_ocde = jaccard_cosine(aiplus_counts, aiplus_tokens, ocde_counts, ocde_tokens)
jaccard_aiplus_pbia, cosine_aiplus_pbia = jaccard_cosine(aiplus_counts, aiplus_tokens, pbia_counts, pbia_tokens)

print(f"{AIPLUS_LABEL} × {BRICS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_aiplus_brics:.3f}")
print(f"  Similaridade de cosseno: {cosine_aiplus_brics:.3f}")
print()
print(f"{AIPLUS_LABEL} × {OECD_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_aiplus_ocde:.3f}")
print(f"  Similaridade de cosseno: {cosine_aiplus_ocde:.3f}")
print()
print(f"{AIPLUS_LABEL} × {PBIA_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_aiplus_pbia:.3f}")
print(f"  Similaridade de cosseno: {cosine_aiplus_pbia:.3f}")


### Gráfico — com qual documento o AI+ se parece mais?

As três comparações lado a lado (AI+ × BRICS, AI+ × OCDE, AI+ × PBIA), para
as duas métricas.

In [ ]:
CATEGORICAL_AIPLUS = "#8b5cf6"  # cor de identidade do AI+ nos gráficos desta seção

metric_labels = ["Índice de Jaccard\n(vocabulário)", "Similaridade de cosseno\n(ênfase)"]
brics_values = [jaccard_aiplus_brics, cosine_aiplus_brics]
ocde_values = [jaccard_aiplus_ocde, cosine_aiplus_ocde]
pbia_values = [jaccard_aiplus_pbia, cosine_aiplus_pbia]

fig, ax = plt.subplots(figsize=(8, 5.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = list(range(len(metric_labels)))
bar_h = 0.24
offsets = [bar_h + 0.03, 0.0, -(bar_h + 0.03)]

series = [
    (brics_values, offsets[0], CATEGORICAL_BLUE, f"{AIPLUS_LABEL} × {BRICS_LABEL}"),
    (ocde_values, offsets[1], CATEGORICAL_AQUA, f"{AIPLUS_LABEL} × {OECD_LABEL}"),
    (pbia_values, offsets[2], CATEGORICAL_PBIA, f"{AIPLUS_LABEL} × {PBIA_LABEL}"),
]

for values, offset, color, label in series:
    ys = [y + offset for y in y_pos]
    ax.barh(ys, values, height=bar_h, color=color, zorder=3, label=label)
    for y, v in zip(ys, values):
        ax.text(v + 0.012, y, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(metric_labels, color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Similaridade (0 a 1)", color=INK_MUTED, fontsize=10)
all_values = brics_values + ocde_values + pbia_values
ax.set_xlim(0, max(all_values) * 1.35)
ax.set_title("Com qual documento o AI+ se parece mais?", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 4.2. O vocabulário mais usado pelo AI+ ecoa mais no BRICS ou na OCDE?

Para os termos mais frequentes do AI+ (excluindo palavras autorreferentes
como *china* e *chinese*), comparamos o quanto **BRICS** e **OCDE** usam
cada um desses mesmos termos — a mesma análise feita para o PBIA na Seção
3.2, agora para a política chinesa.

In [ ]:
SELF_REFERENTIAL_AIPLUS = {"china", "chinese"}
MIN_AIPLUS_COUNT = 3
TOP_N_AIPLUS_TERMS = 18

aiplus_terms_rows = []
for t, aiplus_c in aiplus_counts.most_common():
    if t in SELF_REFERENTIAL_AIPLUS or aiplus_c < MIN_AIPLUS_COUNT:
        continue
    b = relative_freq(brics_counts, len(brics_tokens), t)
    o = relative_freq(ocde_counts, len(ocde_tokens), t)
    aiplus_terms_rows.append((t, b, o, b - o))
    if len(aiplus_terms_rows) >= TOP_N_AIPLUS_TERMS:
        break

aiplus_terms_rows.sort(key=lambda x: x[3])  # OCDE-leaning no topo, BRICS-leaning na base

labels = [r[0] for r in aiplus_terms_rows]
brics_vals = [r[1] for r in aiplus_terms_rows]
ocde_vals = [r[2] for r in aiplus_terms_rows]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(aiplus_terms_rows) + 1), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = range(len(aiplus_terms_rows))
ax.barh(y_pos, [-v for v in brics_vals], color=CATEGORICAL_BLUE, height=0.6, zorder=3, label=BRICS_LABEL)
ax.barh(y_pos, ocde_vals, color=CATEGORICAL_AQUA, height=0.6, zorder=3, label=OECD_LABEL)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9)
ax.axvline(0, color=GRIDLINE, linewidth=1.0, zorder=2)

ax.set_xlabel("Frequência relativa em cada documento (ocorrências por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Vocabulário mais usado pelo AI+: quem mais o ecoa — {BRICS_LABEL} (esquerda) × {OECD_LABEL} (direita)",
    color=INK_PRIMARY, fontsize=12, fontweight="bold", pad=14,
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)

max_abs = max(max(brics_vals), max(ocde_vals), 1e-6)
ax.set_xlim(-max_abs * 1.2, max_abs * 1.2)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.0f}" for x in xticks], color=INK_MUTED, fontsize=9)

ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 4.3. Mapa combinado — PBIA e AI+ juntos entre BRICS e OCDE

Eixo X: similaridade com o **BRICS Declaration**. Eixo Y: similaridade com a
**OECD Recommendation**. Este mapa reaproveita o mapa da Seção 3.3, mas
agora posiciona **os dois documentos nacionais ao mesmo tempo** — PBIA
(Brasil) e AI+ (China) — permitindo comparar diretamente até que ponto cada
país se aproxima mais do eixo de desenvolvimento do BRICS ou do eixo de
governança da OCDE. Como antes, cada documento aparece com dois pontos
(cosseno, o principal, e Jaccard, o secundário, ligados por uma linha
pontilhada), e os cantos de referência marcam onde o próprio BRICS e a
própria OCDE cairiam neste mapa.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

axis_max = 1.05
ax.plot([0, axis_max], [0, axis_max], linestyle="--", linewidth=1.2, color=INK_MUTED, zorder=2)

# Pontos de referência: onde o próprio BRICS e a própria OCDE cairiam neste mapa
ax.scatter([1.0], [cosine_similarity], s=140, color=CATEGORICAL_BLUE, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{BRICS_LABEL} (referência)")
ax.annotate(BRICS_LABEL, (1.0, cosine_similarity), textcoords="offset points", xytext=(-8, 8),
            ha="right", fontsize=8.5, color=INK_MUTED)

ax.scatter([cosine_similarity], [1.0], s=140, color=CATEGORICAL_AQUA, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{OECD_LABEL} (referência)")
ax.annotate(OECD_LABEL, (cosine_similarity, 1.0), textcoords="offset points", xytext=(8, -10),
            ha="left", fontsize=8.5, color=INK_MUTED)

# PBIA — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_pbia_brics, cosine_pbia_brics], [jaccard_pbia_ocde, cosine_pbia_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_PBIA, zorder=4)
ax.scatter([jaccard_pbia_brics], [jaccard_pbia_ocde], s=110, color=CATEGORICAL_PBIA, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="PBIA (Jaccard)")
ax.scatter([cosine_pbia_brics], [cosine_pbia_ocde], s=340, color=CATEGORICAL_PBIA, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="PBIA (cosseno)")
ax.annotate("PBIA", (cosine_pbia_brics, cosine_pbia_ocde), textcoords="offset points", xytext=(10, 10),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

# AI+ — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_aiplus_brics, cosine_aiplus_brics], [jaccard_aiplus_ocde, cosine_aiplus_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_AIPLUS, zorder=4)
ax.scatter([jaccard_aiplus_brics], [jaccard_aiplus_ocde], s=110, color=CATEGORICAL_AIPLUS, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="AI+ (Jaccard)")
ax.scatter([cosine_aiplus_brics], [cosine_aiplus_ocde], s=340, color=CATEGORICAL_AIPLUS, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="AI+ (cosseno)")
ax.annotate("AI+", (cosine_aiplus_brics, cosine_aiplus_ocde), textcoords="offset points", xytext=(10, -14),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

ax.set_xlabel(f"Similaridade com o {BRICS_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_ylabel(f"Similaridade com a {OECD_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_title("PBIA e AI+ no mapa BRICS × OCDE — qual país se aproxima de qual eixo?",
             color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)
ax.set_xlim(0, axis_max)
ax.set_ylim(0, axis_max)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5)

fig.tight_layout()
plt.show()


## Síntese — AI+, PBIA, BRICS e OCDE

- Nas duas métricas, o **AI+ também se aproxima mais do eixo do BRICS do que
  do eixo da OCDE**: similaridade de cosseno de **0,417** com o BRICS contra
  **0,330** com a OCDE, e Jaccard de **0,190** contra **0,155**. O padrão
  repete o observado no PBIA (0,538 × 0,395), embora com similaridade
  absoluta mais baixa nos dois eixos — o AI+ é, dos três documentos
  analisados, o que menos se parece tanto com o BRICS quanto com a OCDE.
- A similaridade do AI+ com o **próprio PBIA** é notavelmente alta: cosseno
  de **0,404** (quase empatado com o BRICS) e Jaccard de **0,205** — o maior
  Jaccard entre os três pares do AI+. Ou seja, em termos de sobreposição
  bruta de vocabulário, o AI+ se parece mais com o plano brasileiro do que
  com a própria declaração do BRICS, mesmo não integrando o bloco.
- O vocabulário mais usado pelo próprio AI+ confirma essa inclinação: termos
  centrais como *development* (32 ocorrências) e *governance* (10) aparecem
  proporcionalmente mais no BRICS do que na OCDE, enquanto termos como
  *systems* e *research* pendem para o lado da OCDE — mas em volume bem
  menor. Muito do vocabulário mais frequente do AI+ (*intelligent*,
  *application*, *smart*, *terminal*, *agents*) é técnico e específico da
  política chinesa, sem equivalente forte em nenhum dos dois documentos de
  referência — o que explica por que as similaridades absolutas do AI+ são
  mais baixas que as do PBIA.
- No **mapa combinado**, tanto o PBIA quanto o AI+ ficam do lado do eixo X
  (mais perto do BRICS do que da OCDE), mas o PBIA está bem mais próximo do
  canto de referência do BRICS do que o AI+. O plano brasileiro absorve mais
  explicitamente a linguagem de desenvolvimento e cooperação do BRICS,
  enquanto a política chinesa, apesar de também pender para esse eixo,
  mantém um vocabulário técnico e de infraestrutura de IA mais próprio, que
  não é plenamente capturado nem pelo eixo do BRICS nem pelo da OCDE.

## 5. Posicionamento do Brasil (PBIA) diante do BRICS, da OCDE e do AI+

As seções anteriores compararam os documentos dois a dois. Esta seção reúne
as seis similaridades de cosseno já calculadas (BRICS×OCDE, BRICS×AI+,
BRICS×PBIA, OCDE×AI+, OCDE×PBIA, AI+×PBIA) em duas visualizações de síntese,
de leitura acadêmica consolidada:

1. **Gráfico radar** — o perfil de similaridade do PBIA nos três eixos
   (BRICS, OCDE, AI+), comparado a uma linha de referência: a similaridade
   *média* que BRICS, OCDE e AI+ têm entre si. Isso responde: o Brasil está
   mais ou menos alinhado a cada bloco do que os próprios blocos
   internacionais estão entre si?
2. **Mapa de escalonamento multidimensional (MDS clássico)** — projeta os
   quatro documentos em um plano 2D a partir da matriz completa de
   distâncias (1 − similaridade de cosseno), preservando o quanto possível
   as distâncias relativas originais. É a técnica padrão em linguística de
   corpus e cientometria para visualizar espaços de similaridade textual.

### 5.1. Gráfico radar — perfil de alinhamento do PBIA

Cada eixo é a similaridade de cosseno do PBIA com um documento de
referência. A linha tracejada cinza é a **similaridade média entre os
próprios três documentos internacionais** nesse mesmo eixo (ex.: no eixo
BRICS, a média de OCDE×BRICS e AI+×BRICS) — o "quanto os blocos
internacionais se parecem entre si", usado como referência de comparação
para o perfil do Brasil.

In [ ]:
import numpy as np

radar_categories = [BRICS_LABEL, OECD_LABEL, AIPLUS_LABEL]

pbia_profile = [cosine_pbia_brics, cosine_pbia_ocde, cosine_aiplus_pbia]

# Referência: para cada eixo, a média de similaridade entre os OUTROS dois
# documentos internacionais (ex.: no eixo BRICS, a média de OCDE×BRICS e AI+×BRICS)
international_baseline = [
    (cosine_similarity + cosine_aiplus_brics) / 2,   # eixo BRICS: OCDE×BRICS e AI+×BRICS
    (cosine_similarity + cosine_aiplus_ocde) / 2,    # eixo OCDE: BRICS×OCDE e AI+×OCDE
    (cosine_aiplus_brics + cosine_aiplus_ocde) / 2,  # eixo AI+: BRICS×AI+ e OCDE×AI+
]

print("Perfil PBIA:            ", [f"{v:.3f}" for v in pbia_profile])
print("Referência internacional:", [f"{v:.3f}" for v in international_baseline])

n_axes = len(radar_categories)
angles = [n / float(n_axes) * 2 * np.pi for n in range(n_axes)]
angles += angles[:1]

pbia_plot = pbia_profile + pbia_profile[:1]
baseline_plot = international_baseline + international_baseline[:1]

fig, ax = plt.subplots(figsize=(7.5, 7.5), subplot_kw=dict(polar=True), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_categories, color=INK_PRIMARY, fontsize=11)

max_r = max(pbia_plot + baseline_plot) * 1.25
ax.set_ylim(0, max_r)
r_ticks = np.round(np.linspace(0, max_r, 5), 2)
ax.set_yticks(r_ticks)
ax.set_yticklabels([f"{r:.2f}" for r in r_ticks], color=INK_MUTED, fontsize=8)
ax.tick_params(axis="x", pad=12)

ax.spines["polar"].set_color(GRIDLINE)
ax.grid(color=GRIDLINE, linewidth=0.8)

ax.plot(angles, baseline_plot, linestyle="--", linewidth=1.4, color=INK_MUTED,
        label="Média entre os 3 documentos internacionais")
ax.fill(angles, baseline_plot, color=INK_MUTED, alpha=0.06)

ax.plot(angles, pbia_plot, linewidth=2.2, color=CATEGORICAL_PBIA, label=PBIA_LABEL)
ax.fill(angles, pbia_plot, color=CATEGORICAL_PBIA, alpha=0.18)
ax.scatter(angles, pbia_plot, s=50, color=CATEGORICAL_PBIA, zorder=5, edgecolors=SURFACE, linewidths=1.0)

for angle, value in zip(angles, pbia_plot):
    ax.annotate(f"{value:.3f}", (angle, value), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=9, color=INK_PRIMARY, fontweight="bold")

ax.set_title(
    "Perfil de alinhamento do PBIA — similaridade de cosseno com BRICS, OCDE e AI+",
    color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=28,
)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 5.2. Mapa de escalonamento multidimensional (MDS) — os quatro documentos no mesmo plano

Construímos a matriz completa de distâncias entre os quatro documentos
(distância = 1 − similaridade de cosseno) e aplicamos o **MDS clássico**
(Torgerson): dupla centralização da matriz de distâncias ao quadrado,
seguida de decomposição em autovalores/autovetores, mantendo os dois
componentes de maior variância. O resultado é um mapa 2D em que a distância
entre dois pontos aproxima, o melhor possível, a dissimilaridade textual
real entre os dois documentos — sem impor de antemão nenhum eixo temático
(diferente do mapa BRICS × OCDE das seções 3.3/4.3, aqui nenhum documento é
usado como eixo de referência).

In [ ]:
doc_labels = [BRICS_LABEL, OECD_LABEL, AIPLUS_LABEL, PBIA_LABEL]
doc_colors = [CATEGORICAL_BLUE, CATEGORICAL_AQUA, CATEGORICAL_AIPLUS, CATEGORICAL_PBIA]

# Matriz de similaridade de cosseno (ordem: BRICS, OCDE, AI+, PBIA)
S = np.array([
    [1.0,                cosine_similarity,     cosine_aiplus_brics, cosine_pbia_brics],
    [cosine_similarity,  1.0,                   cosine_aiplus_ocde,  cosine_pbia_ocde],
    [cosine_aiplus_brics, cosine_aiplus_ocde,   1.0,                 cosine_aiplus_pbia],
    [cosine_pbia_brics,  cosine_pbia_ocde,      cosine_aiplus_pbia,  1.0],
])
D = 1.0 - S
np.fill_diagonal(D, 0.0)

def classical_mds(dist_matrix, n_components=2):
    n = dist_matrix.shape[0]
    D2 = dist_matrix ** 2
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D2 @ J
    eigvals, eigvecs = np.linalg.eigh(B)
    order = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    eigvals_pos = np.clip(eigvals, 0, None)
    coords = eigvecs[:, :n_components] * np.sqrt(eigvals_pos[:n_components])
    explained = eigvals_pos[:n_components].sum() / eigvals_pos.sum()
    return coords, explained

coords, explained_variance = classical_mds(D)
print("Coordenadas MDS (dim 1, dim 2):")
for label, (x, y) in zip(doc_labels, coords):
    print(f"  {label:24s} ({x:+.3f}, {y:+.3f})")
print(f"\nVariância explicada pelos 2 componentes: {explained_variance*100:.1f}%")

fig, ax = plt.subplots(figsize=(7.5, 7.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

# Linhas conectando todos os pares, com espessura proporcional à similaridade
# (par mais similar = linha mais grossa e mais opaca)
for i in range(len(doc_labels)):
    for j in range(i + 1, len(doc_labels)):
        sim = S[i, j]
        ax.plot([coords[i, 0], coords[j, 0]], [coords[i, 1], coords[j, 1]],
                color=INK_MUTED, linewidth=0.6 + sim * 3.2, alpha=0.18 + sim * 0.35, zorder=1)

for label, color, (x, y) in zip(doc_labels, doc_colors, coords):
    ax.scatter([x], [y], s=420, color=color, alpha=0.9, edgecolors=SURFACE,
               linewidths=1.6, zorder=3)
    ax.annotate(label, (x, y), textcoords="offset points", xytext=(0, 20),
                ha="center", fontsize=11, fontweight="bold", color=INK_PRIMARY, zorder=4)

ax.axhline(0, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.axvline(0, color=GRIDLINE, linewidth=0.8, zorder=0)

pad = max(np.abs(coords).max() * 0.45, 0.05)
ax.set_xlim(coords[:, 0].min() - pad, coords[:, 0].max() + pad)
ax.set_ylim(coords[:, 1].min() - pad, coords[:, 1].max() + pad)

ax.set_xlabel("Dimensão MDS 1", color=INK_MUTED, fontsize=10)
ax.set_ylabel("Dimensão MDS 2", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Mapa de similaridade textual dos quatro documentos (MDS clássico, {explained_variance*100:.0f}% da variância)",
    color=INK_PRIMARY, fontsize=12.5, fontweight="bold", pad=16,
)

for spine in ax.spines.values():
    spine.set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()


### Síntese — o posicionamento do Brasil

- No **gráfico radar**, o PBIA supera a similaridade média internacional
  nos eixos **BRICS** (0,538 vs. 0,468) e **AI+** (0,404 vs. 0,374), e fica
  ligeiramente abaixo dela apenas no eixo **OCDE** (0,395 vs. 0,425). Ou
  seja, o Brasil está **mais próximo do BRICS e do AI+ do que os documentos
  internacionais estão, em média, uns dos outros**, e apenas discretamente
  mais distante da OCDE do que a média do próprio grupo internacional — um
  desalinhamento pequeno, não um afastamento acentuado.
- No **mapa MDS**, a ligação **BRICS–PBIA (cosseno 0,538)** é o par mais
  próximo de todo o conjunto de quatro documentos — mais próximo até do que
  o par BRICS–OCDE (0,519), que serviu de referência nas seções anteriores.
  O BRICS funciona como um "elo": é o documento mais parecido tanto com o
  PBIA quanto com a OCDE. Já o par **OCDE–AI+ (0,330)** é o mais distante de
  todo o estudo — a política chinesa e a recomendação da OCDE são, em termos
  de vocabulário e ênfase, os dois documentos mais diferentes entre si.
- O PBIA fica claramente puxado para perto do BRICS no mapa, enquanto suas
  distâncias até a OCDE (0,605) e até o AI+ (0,583) são parecidas entre
  si — ou seja, o Brasil não está simplesmente "afastado da OCDE": ele tem
  uma afinidade específica e mensurável com o BRICS, e uma distância
  semelhante tanto da OCDE quanto do AI+, os dois documentos mais técnicos
  e "institucionais" do conjunto.
- Em conjunto, as duas visualizações sustentam a mesma conclusão por
  caminhos metodológicos independentes (perfil radial vs. projeção
  geométrica de distâncias, que preservou fielmente a ordem de todas as
  seis distâncias par a par): a política brasileira de IA (PBIA) se
  posiciona, na sua ênfase textual, mais próxima da agenda de
  desenvolvimento tecnológico nacional do BRICS do que de qualquer outro
  documento do estudo — inclusive do AI+, apesar de este também pender
  para o eixo BRICS quando comparado isoladamente com a OCDE.

## 6. New Generation AI Development Plan (China New Gen) — a qual documento o plano chinês se aproxima mais?

O **China New Gen** (`China_New_Gen.json`) é o *State Council Notice on the
Issuance of the New Generation Artificial Intelligence Development Plan*
(2017), o plano-mestre de longo prazo da China para IA — anterior e mais
amplo que o AI+ (que é uma política setorial de 2023 de aprofundamento da
implementação). Diferente dos demais documentos, o JSON deste plano guarda o
texto em uma árvore (`sections` → capítulos → subseções → itens numerados →
boxes, cada nível com `title` e `paragraphs`), em vez de uma lista simples de
seções com campo `text`. Por isso, ele usa uma função de carregamento própria
que percorre essa árvore recursivamente, mas aplica exatamente o mesmo filtro
de tokenização (mesma lista de *stopwords*, mesmo filtro de tamanho mínimo de
palavra) usado para BRICS, OCDE, PBIA e AI+, para que todas as séries de
frequência continuem diretamente comparáveis.

Comparamos o China New Gen com os quatro documentos já carregados — BRICS
Declaration, OECD Recommendation, PBIA e AI+ — para responder: o vocabulário
e a ênfase temática do plano chinês de longo prazo se aproximam mais do eixo
**desenvolvimento / países em desenvolvimento / economia digital** (BRICS),
do eixo **arquitetura de governança / atores responsáveis** (OCDE), do plano
brasileiro (PBIA), ou da política setorial mais recente da própria China
(AI+)?

In [ ]:
CHINA_LABEL = "China New Gen"

def load_tokens_nested(path):
    """Carrega tokens de um JSON com texto em árvore (sections -> capítulos ->
    subseções -> itens -> boxes), como o China_New_Gen.json, em vez da lista
    simples de seções com campo "text" usada pelos demais documentos."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    parts = [data["document"]["title"], data["document"].get("preamble", "")]

    def walk(node, parts):
        if "title" in node:
            parts.append(node["title"])
        if "paragraphs" in node:
            parts.extend(node["paragraphs"])
        for key in ("subsections", "items", "boxes"):
            for child in node.get(key, []):
                walk(child, parts)

    for chapter in data["sections"]:
        walk(chapter, parts)

    text = " ".join(parts)
    tokens = re.findall(r"[a-zA-Z]+(?:-[a-zA-Z]+)*", text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2], data

china_tokens, china_data = load_tokens_nested("China_New_Gen.json")
china_counts = Counter(china_tokens)

print(f"{CHINA_LABEL}: {len(china_tokens)} tokens, {len(china_counts)} termos únicos")


### 6.1. Similaridade do China New Gen com cada documento

Reaproveitamos o índice de Jaccard e a similaridade de cosseno definidos na
Seção 1, agora aplicados aos pares (China New Gen, BRICS), (China New Gen,
OCDE), (China New Gen, PBIA) e (China New Gen, AI+).

In [ ]:
jaccard_china_brics, cosine_china_brics = jaccard_cosine(china_counts, china_tokens, brics_counts, brics_tokens)
jaccard_china_ocde, cosine_china_ocde = jaccard_cosine(china_counts, china_tokens, ocde_counts, ocde_tokens)
jaccard_china_pbia, cosine_china_pbia = jaccard_cosine(china_counts, china_tokens, pbia_counts, pbia_tokens)
jaccard_china_aiplus, cosine_china_aiplus = jaccard_cosine(china_counts, china_tokens, aiplus_counts, aiplus_tokens)

print(f"{CHINA_LABEL} × {BRICS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_china_brics:.3f}")
print(f"  Similaridade de cosseno: {cosine_china_brics:.3f}")
print()
print(f"{CHINA_LABEL} × {OECD_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_china_ocde:.3f}")
print(f"  Similaridade de cosseno: {cosine_china_ocde:.3f}")
print()
print(f"{CHINA_LABEL} × {PBIA_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_china_pbia:.3f}")
print(f"  Similaridade de cosseno: {cosine_china_pbia:.3f}")
print()
print(f"{CHINA_LABEL} × {AIPLUS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_china_aiplus:.3f}")
print(f"  Similaridade de cosseno: {cosine_china_aiplus:.3f}")


### Gráfico — com qual documento o China New Gen se parece mais?

As quatro comparações lado a lado (China New Gen × BRICS, × OCDE, × PBIA,
× AI+), para as duas métricas.

In [ ]:
CATEGORICAL_CHINA = "#d6336c"  # cor de identidade do China New Gen nos gráficos desta seção

metric_labels = ["Índice de Jaccard\n(vocabulário)", "Similaridade de cosseno\n(ênfase)"]
brics_values = [jaccard_china_brics, cosine_china_brics]
ocde_values = [jaccard_china_ocde, cosine_china_ocde]
pbia_values = [jaccard_china_pbia, cosine_china_pbia]
aiplus_values = [jaccard_china_aiplus, cosine_china_aiplus]

fig, ax = plt.subplots(figsize=(8, 6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = list(range(len(metric_labels)))
bar_h = 0.18
offsets = [1.5 * bar_h + 0.03, 0.5 * bar_h + 0.01, -(0.5 * bar_h + 0.01), -(1.5 * bar_h + 0.03)]

series = [
    (brics_values, offsets[0], CATEGORICAL_BLUE, f"{CHINA_LABEL} × {BRICS_LABEL}"),
    (ocde_values, offsets[1], CATEGORICAL_AQUA, f"{CHINA_LABEL} × {OECD_LABEL}"),
    (pbia_values, offsets[2], CATEGORICAL_PBIA, f"{CHINA_LABEL} × {PBIA_LABEL}"),
    (aiplus_values, offsets[3], CATEGORICAL_AIPLUS, f"{CHINA_LABEL} × {AIPLUS_LABEL}"),
]

for values, offset, color, label in series:
    ys = [y + offset for y in y_pos]
    ax.barh(ys, values, height=bar_h, color=color, zorder=3, label=label)
    for y, v in zip(ys, values):
        ax.text(v + 0.012, y, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(metric_labels, color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Similaridade (0 a 1)", color=INK_MUTED, fontsize=10)
all_values = brics_values + ocde_values + pbia_values + aiplus_values
ax.set_xlim(0, max(all_values) * 1.35)
ax.set_title("Com qual documento o China New Gen se parece mais?", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 6.2. O vocabulário mais usado pelo China New Gen ecoa mais no BRICS ou na OCDE?

Para os termos mais frequentes do China New Gen (excluindo palavras
autorreferentes como *china* e *chinese*), comparamos o quanto **BRICS** e
**OCDE** usam cada um desses mesmos termos — a mesma análise feita para o
PBIA na Seção 3.2 e para o AI+ na Seção 4.2, agora para o plano-mestre
chinês de longo prazo.

In [ ]:
SELF_REFERENTIAL_CHINA = {"china", "chinese"}
MIN_CHINA_COUNT = 3
TOP_N_CHINA_TERMS = 18

china_terms_rows = []
for t, china_c in china_counts.most_common():
    if t in SELF_REFERENTIAL_CHINA or china_c < MIN_CHINA_COUNT:
        continue
    b = relative_freq(brics_counts, len(brics_tokens), t)
    o = relative_freq(ocde_counts, len(ocde_tokens), t)
    china_terms_rows.append((t, b, o, b - o))
    if len(china_terms_rows) >= TOP_N_CHINA_TERMS:
        break

china_terms_rows.sort(key=lambda x: x[3])  # OCDE-leaning no topo, BRICS-leaning na base

labels = [r[0] for r in china_terms_rows]
brics_vals = [r[1] for r in china_terms_rows]
ocde_vals = [r[2] for r in china_terms_rows]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(china_terms_rows) + 1), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = range(len(china_terms_rows))
ax.barh(y_pos, [-v for v in brics_vals], color=CATEGORICAL_BLUE, height=0.6, zorder=3, label=BRICS_LABEL)
ax.barh(y_pos, ocde_vals, color=CATEGORICAL_AQUA, height=0.6, zorder=3, label=OECD_LABEL)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9)
ax.axvline(0, color=GRIDLINE, linewidth=1.0, zorder=2)

ax.set_xlabel("Frequência relativa em cada documento (ocorrências por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Vocabulário mais usado pelo China New Gen: quem mais o ecoa — {BRICS_LABEL} (esquerda) × {OECD_LABEL} (direita)",
    color=INK_PRIMARY, fontsize=12, fontweight="bold", pad=14,
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)

max_abs = max(max(brics_vals), max(ocde_vals), 1e-6)
ax.set_xlim(-max_abs * 1.2, max_abs * 1.2)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.0f}" for x in xticks], color=INK_MUTED, fontsize=9)

ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 6.3. Mapa combinado — PBIA, AI+ e China New Gen juntos entre BRICS e OCDE

Eixo X: similaridade com o **BRICS Declaration**. Eixo Y: similaridade com a
**OECD Recommendation**. Este mapa estende o da Seção 4.3, agora posicionando
os **três documentos nacionais/de longo prazo ao mesmo tempo** — PBIA
(Brasil), AI+ (China, política setorial de 2023) e China New Gen (China,
plano-mestre de 2017) — no mesmo eixo BRICS × OCDE. Como antes, cada
documento aparece com dois pontos (cosseno, o principal, e Jaccard, o
secundário, ligados por uma linha pontilhada), e os cantos de referência
marcam onde o próprio BRICS e a própria OCDE cairiam neste mapa.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

axis_max = 1.05
ax.plot([0, axis_max], [0, axis_max], linestyle="--", linewidth=1.2, color=INK_MUTED, zorder=2)

# Pontos de referência: onde o próprio BRICS e a própria OCDE cairiam neste mapa
ax.scatter([1.0], [cosine_similarity], s=140, color=CATEGORICAL_BLUE, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{BRICS_LABEL} (referência)")
ax.annotate(BRICS_LABEL, (1.0, cosine_similarity), textcoords="offset points", xytext=(-8, 8),
            ha="right", fontsize=8.5, color=INK_MUTED)

ax.scatter([cosine_similarity], [1.0], s=140, color=CATEGORICAL_AQUA, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{OECD_LABEL} (referência)")
ax.annotate(OECD_LABEL, (cosine_similarity, 1.0), textcoords="offset points", xytext=(8, -10),
            ha="left", fontsize=8.5, color=INK_MUTED)

# PBIA — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_pbia_brics, cosine_pbia_brics], [jaccard_pbia_ocde, cosine_pbia_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_PBIA, zorder=4)
ax.scatter([jaccard_pbia_brics], [jaccard_pbia_ocde], s=110, color=CATEGORICAL_PBIA, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="PBIA (Jaccard)")
ax.scatter([cosine_pbia_brics], [cosine_pbia_ocde], s=340, color=CATEGORICAL_PBIA, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="PBIA (cosseno)")
ax.annotate("PBIA", (cosine_pbia_brics, cosine_pbia_ocde), textcoords="offset points", xytext=(10, 10),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

# AI+ — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_aiplus_brics, cosine_aiplus_brics], [jaccard_aiplus_ocde, cosine_aiplus_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_AIPLUS, zorder=4)
ax.scatter([jaccard_aiplus_brics], [jaccard_aiplus_ocde], s=110, color=CATEGORICAL_AIPLUS, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="AI+ (Jaccard)")
ax.scatter([cosine_aiplus_brics], [cosine_aiplus_ocde], s=340, color=CATEGORICAL_AIPLUS, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="AI+ (cosseno)")
ax.annotate("AI+", (cosine_aiplus_brics, cosine_aiplus_ocde), textcoords="offset points", xytext=(10, -14),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

# China New Gen — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_china_brics, cosine_china_brics], [jaccard_china_ocde, cosine_china_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_CHINA, zorder=4)
ax.scatter([jaccard_china_brics], [jaccard_china_ocde], s=110, color=CATEGORICAL_CHINA, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="China New Gen (Jaccard)")
ax.scatter([cosine_china_brics], [cosine_china_ocde], s=340, color=CATEGORICAL_CHINA, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="China New Gen (cosseno)")
ax.annotate("China New Gen", (cosine_china_brics, cosine_china_ocde), textcoords="offset points", xytext=(10, -14),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

ax.set_xlabel(f"Similaridade com o {BRICS_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_ylabel(f"Similaridade com a {OECD_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_title("PBIA, AI+ e China New Gen no mapa BRICS × OCDE — qual país/plano se aproxima de qual eixo?",
             color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)
ax.set_xlim(0, axis_max)
ax.set_ylim(0, axis_max)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5)

fig.tight_layout()
plt.show()


### Síntese — China New Gen, BRICS, OCDE, PBIA e AI+

- O China New Gen também pende para o eixo do **BRICS** em vez do eixo da
  **OCDE**, replicando o padrão observado no PBIA e no AI+ — os três
  documentos nacionais do estudo se aproximam mais da ênfase em
  desenvolvimento tecnológico do BRICS do que da arquitetura de governança
  da OCDE.
- A maior similaridade do China New Gen, no entanto, é com o **AI+**: os
  dois documentos são políticas chinesas de IA (separadas por seis anos, o
  plano-mestre de 2017 e o aprofundamento setorial de 2023) e compartilham
  vocabulário técnico específico — *intelligent*, *smart*, *technology*,
  *innovation*, *systems* — não replicado nos demais documentos do estudo.
- No **mapa combinado**, o China New Gen aparece deslocado dos outros dois
  pontos nacionais (PBIA e AI+), mais próximo da diagonal BRICS=OCDE — sinal
  de que, apesar de também pender para o BRICS, o plano chinês de longo
  prazo tem uma similaridade absoluta menor com os dois blocos do que PBIA e
  AI+, refletindo seu vocabulário mais técnico e menos político/diplomático
  (o texto é dominado por metas de P&D, infraestrutura e insdústria, não por
  linguagem de cooperação internacional ou governança).

## 7. Síntese final — os cinco documentos juntos

Estendemos o mapa de escalonamento multidimensional (MDS) da Seção 5.2 para
incluir o **China New Gen** como um quinto documento, ao lado de BRICS, OCDE,
AI+ e PBIA. Construímos a matriz completa de distâncias entre os cinco
documentos (distância = 1 − similaridade de cosseno) e reaproveitamos a
função `classical_mds` já definida na Seção 5.2, mantendo os dois componentes
de maior variância — sem impor de antemão nenhum eixo temático, ao contrário
dos mapas BRICS × OCDE das seções 4.3/6.3.

In [ ]:
doc_labels5 = [BRICS_LABEL, OECD_LABEL, AIPLUS_LABEL, PBIA_LABEL, CHINA_LABEL]
doc_colors5 = [CATEGORICAL_BLUE, CATEGORICAL_AQUA, CATEGORICAL_AIPLUS, CATEGORICAL_PBIA, CATEGORICAL_CHINA]

# Matriz de similaridade de cosseno (ordem: BRICS, OCDE, AI+, PBIA, China New Gen)
S5 = np.array([
    [1.0,                 cosine_similarity,    cosine_aiplus_brics, cosine_pbia_brics, cosine_china_brics],
    [cosine_similarity,   1.0,                  cosine_aiplus_ocde,  cosine_pbia_ocde,  cosine_china_ocde],
    [cosine_aiplus_brics, cosine_aiplus_ocde,   1.0,                 cosine_aiplus_pbia, cosine_china_aiplus],
    [cosine_pbia_brics,   cosine_pbia_ocde,     cosine_aiplus_pbia,  1.0,                cosine_china_pbia],
    [cosine_china_brics,  cosine_china_ocde,    cosine_china_aiplus, cosine_china_pbia,  1.0],
])
D5 = 1.0 - S5
np.fill_diagonal(D5, 0.0)

coords5, explained_variance5 = classical_mds(D5)
print("Coordenadas MDS (dim 1, dim 2):")
for label, (x, y) in zip(doc_labels5, coords5):
    print(f"  {label:24s} ({x:+.3f}, {y:+.3f})")
print(f"\nVariância explicada pelos 2 componentes: {explained_variance5*100:.1f}%")

fig, ax = plt.subplots(figsize=(8, 8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

# Linhas conectando todos os pares, com espessura proporcional à similaridade
# (par mais similar = linha mais grossa e mais opaca)
for i in range(len(doc_labels5)):
    for j in range(i + 1, len(doc_labels5)):
        sim = S5[i, j]
        ax.plot([coords5[i, 0], coords5[j, 0]], [coords5[i, 1], coords5[j, 1]],
                color=INK_MUTED, linewidth=0.6 + sim * 3.2, alpha=0.18 + sim * 0.35, zorder=1)

for label, color, (x, y) in zip(doc_labels5, doc_colors5, coords5):
    ax.scatter([x], [y], s=420, color=color, alpha=0.9, edgecolors=SURFACE,
               linewidths=1.6, zorder=3)
    ax.annotate(label, (x, y), textcoords="offset points", xytext=(0, 20),
                ha="center", fontsize=11, fontweight="bold", color=INK_PRIMARY, zorder=4)

ax.axhline(0, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.axvline(0, color=GRIDLINE, linewidth=0.8, zorder=0)

pad = max(np.abs(coords5).max() * 0.45, 0.05)
ax.set_xlim(coords5[:, 0].min() - pad, coords5[:, 0].max() + pad)
ax.set_ylim(coords5[:, 1].min() - pad, coords5[:, 1].max() + pad)

ax.set_xlabel("Dimensão MDS 1", color=INK_MUTED, fontsize=10)
ax.set_ylabel("Dimensão MDS 2", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Mapa de similaridade textual dos cinco documentos (MDS clássico, {explained_variance5*100:.0f}% da variância)",
    color=INK_PRIMARY, fontsize=12.5, fontweight="bold", pad=16,
)

for spine in ax.spines.values():
    spine.set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()


### Síntese — o mapa completo dos cinco documentos

- A ligação mais forte de **todo** o mapa de cinco documentos passa a ser
  **China New Gen–AI+ (cosseno 0,742)** — muito acima de qualquer par
  observado até aqui, inclusive do antigo campeão **BRICS–PBIA (0,538)**, que
  era o par mais próximo antes da entrada do China New Gen (Seção 5.2). Isso
  evidencia que a proximidade de autoria — duas políticas chinesas de IA,
  mesmo separadas por seis anos e por escopos diferentes — supera qualquer
  proximidade de bloco político observada no restante do estudo, e no mapa
  aparece como o par de pontos visivelmente mais próximo entre si.
- Logo atrás, o restante da hierarquia de proximidade observada na Seção 5
  se mantém: **BRICS–PBIA (0,538)** e **BRICS–OCDE (0,519)** seguem como o
  segundo e o terceiro pares mais próximos, com o **BRICS permanecendo o
  "elo"** do conjunto — o documento mais parecido com praticamente todos os
  outros.
- O par mais distante de todo o mapa continua sendo **AI+–OCDE (0,330)**,
  seguido de perto por **China New Gen–OCDE (0,379)** — as duas políticas
  chinesas são, das cinco, as que menos dialogam com a linguagem de
  governança, atores e responsabilidade da OCDE.
- O **China New Gen** ocupa uma posição própria no mapa, junto ao AI+ (dado
  o par mais forte do estudo) mas puxado para mais perto do centro do que
  ele — reflexo de uma similaridade absoluta um pouco maior com BRICS e PBIA
  do que a que o AI+ tem com os mesmos dois documentos (0,447 e 0,475 do
  China New Gen contra 0,417 e 0,404 do AI+).

## 8. Apply AI Strategy (UE) — a qual documento a estratégia europeia se aproxima mais?

O **Apply AI Strategy** (`Apply_AI_Strategy.json`) é a comunicação da Comissão
Europeia *Apply AI Strategy* (COM(2025) 723 final, out/2025) — a estratégia da
UE para acelerar a adoção setorial de IA (saúde, manufatura, defesa,
mobilidade, setor público, etc.). Assim como BRICS, OCDE, PBIA e AI+, este
JSON guarda o texto em uma lista simples de seções com campo `text`, então
reaproveitamos diretamente a função `load_tokens` da Seção 1 (mesma lista de
*stopwords*, mesmo filtro de tamanho mínimo de palavra).

Comparamos o Apply AI Strategy com os cinco documentos já carregados — BRICS
Declaration, OECD Recommendation, PBIA, AI+ e China New Gen — para responder:
sendo a UE um bloco de países desenvolvidos com forte tradição regulatória,
o vocabulário e a ênfase temática do Apply AI Strategy se aproximam mais do
eixo **arquitetura de governança / atores responsáveis** da OCDE, ou do eixo
**desenvolvimento / economia digital** do BRICS — invertendo o padrão visto
em PBIA, AI+ e China New Gen, os três documentos que até aqui penderam para
o BRICS?

In [ ]:
APPLY_LABEL = "Apply AI Strategy (UE)"

apply_tokens, apply_data = load_tokens("Apply_AI_Strategy.json")
apply_counts = Counter(apply_tokens)

print(f"{APPLY_LABEL}: {len(apply_tokens)} tokens, {len(apply_counts)} termos únicos")


### 8.1. Similaridade do Apply AI Strategy com cada documento

Reaproveitamos o índice de Jaccard e a similaridade de cosseno definidos na
Seção 1, agora aplicados aos pares (Apply AI Strategy, BRICS), (Apply AI
Strategy, OCDE), (Apply AI Strategy, PBIA), (Apply AI Strategy, AI+) e
(Apply AI Strategy, China New Gen).

In [ ]:
jaccard_apply_brics, cosine_apply_brics = jaccard_cosine(apply_counts, apply_tokens, brics_counts, brics_tokens)
jaccard_apply_ocde, cosine_apply_ocde = jaccard_cosine(apply_counts, apply_tokens, ocde_counts, ocde_tokens)
jaccard_apply_pbia, cosine_apply_pbia = jaccard_cosine(apply_counts, apply_tokens, pbia_counts, pbia_tokens)
jaccard_apply_aiplus, cosine_apply_aiplus = jaccard_cosine(apply_counts, apply_tokens, aiplus_counts, aiplus_tokens)
jaccard_apply_china, cosine_apply_china = jaccard_cosine(apply_counts, apply_tokens, china_counts, china_tokens)

print(f"{APPLY_LABEL} × {BRICS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_apply_brics:.3f}")
print(f"  Similaridade de cosseno: {cosine_apply_brics:.3f}")
print()
print(f"{APPLY_LABEL} × {OECD_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_apply_ocde:.3f}")
print(f"  Similaridade de cosseno: {cosine_apply_ocde:.3f}")
print()
print(f"{APPLY_LABEL} × {PBIA_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_apply_pbia:.3f}")
print(f"  Similaridade de cosseno: {cosine_apply_pbia:.3f}")
print()
print(f"{APPLY_LABEL} × {AIPLUS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_apply_aiplus:.3f}")
print(f"  Similaridade de cosseno: {cosine_apply_aiplus:.3f}")
print()
print(f"{APPLY_LABEL} × {CHINA_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_apply_china:.3f}")
print(f"  Similaridade de cosseno: {cosine_apply_china:.3f}")


### Gráfico — com qual documento o Apply AI Strategy se parece mais?

As cinco comparações lado a lado (Apply AI Strategy × BRICS, × OCDE, × PBIA,
× AI+, × China New Gen), para as duas métricas.

In [ ]:
CATEGORICAL_APPLY = "#f59f00"  # cor de identidade do Apply AI Strategy nos gráficos desta seção

metric_labels = ["Índice de Jaccard\n(vocabulário)", "Similaridade de cosseno\n(ênfase)"]
brics_values = [jaccard_apply_brics, cosine_apply_brics]
ocde_values = [jaccard_apply_ocde, cosine_apply_ocde]
pbia_values = [jaccard_apply_pbia, cosine_apply_pbia]
aiplus_values = [jaccard_apply_aiplus, cosine_apply_aiplus]
china_values = [jaccard_apply_china, cosine_apply_china]

fig, ax = plt.subplots(figsize=(8, 6.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = list(range(len(metric_labels)))
bar_h = 0.15
offsets = [2 * bar_h + 0.04, bar_h + 0.02, 0.0, -(bar_h + 0.02), -(2 * bar_h + 0.04)]

series = [
    (brics_values, offsets[0], CATEGORICAL_BLUE, f"{APPLY_LABEL} × {BRICS_LABEL}"),
    (ocde_values, offsets[1], CATEGORICAL_AQUA, f"{APPLY_LABEL} × {OECD_LABEL}"),
    (pbia_values, offsets[2], CATEGORICAL_PBIA, f"{APPLY_LABEL} × {PBIA_LABEL}"),
    (aiplus_values, offsets[3], CATEGORICAL_AIPLUS, f"{APPLY_LABEL} × {AIPLUS_LABEL}"),
    (china_values, offsets[4], CATEGORICAL_CHINA, f"{APPLY_LABEL} × {CHINA_LABEL}"),
]

for values, offset, color, label in series:
    ys = [y + offset for y in y_pos]
    ax.barh(ys, values, height=bar_h, color=color, zorder=3, label=label)
    for y, v in zip(ys, values):
        ax.text(v + 0.012, y, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(metric_labels, color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Similaridade (0 a 1)", color=INK_MUTED, fontsize=10)
all_values = brics_values + ocde_values + pbia_values + aiplus_values + china_values
ax.set_xlim(0, max(all_values) * 1.35)
ax.set_title("Com qual documento o Apply AI Strategy se parece mais?", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 8.2. O vocabulário mais usado pelo Apply AI Strategy ecoa mais no BRICS ou na OCDE?

Para os termos mais frequentes do Apply AI Strategy (excluindo palavras
autorreferentes como *european*, *europe*, *commission* e *eu*), comparamos o
quanto **BRICS** e **OCDE** usam cada um desses mesmos termos — a mesma
análise feita para PBIA, AI+ e China New Gen nas seções anteriores, agora
para a estratégia europeia.

In [ ]:
SELF_REFERENTIAL_APPLY = {"european", "europe", "commission", "eu"}
MIN_APPLY_COUNT = 3
TOP_N_APPLY_TERMS = 18

apply_terms_rows = []
for t, apply_c in apply_counts.most_common():
    if t in SELF_REFERENTIAL_APPLY or apply_c < MIN_APPLY_COUNT:
        continue
    b = relative_freq(brics_counts, len(brics_tokens), t)
    o = relative_freq(ocde_counts, len(ocde_tokens), t)
    apply_terms_rows.append((t, b, o, b - o))
    if len(apply_terms_rows) >= TOP_N_APPLY_TERMS:
        break

apply_terms_rows.sort(key=lambda x: x[3])  # OCDE-leaning no topo, BRICS-leaning na base

labels = [r[0] for r in apply_terms_rows]
brics_vals = [r[1] for r in apply_terms_rows]
ocde_vals = [r[2] for r in apply_terms_rows]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(apply_terms_rows) + 1), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = range(len(apply_terms_rows))
ax.barh(y_pos, [-v for v in brics_vals], color=CATEGORICAL_BLUE, height=0.6, zorder=3, label=BRICS_LABEL)
ax.barh(y_pos, ocde_vals, color=CATEGORICAL_AQUA, height=0.6, zorder=3, label=OECD_LABEL)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9)
ax.axvline(0, color=GRIDLINE, linewidth=1.0, zorder=2)

ax.set_xlabel("Frequência relativa em cada documento (ocorrências por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Vocabulário mais usado pelo Apply AI Strategy: quem mais o ecoa — {BRICS_LABEL} (esquerda) × {OECD_LABEL} (direita)",
    color=INK_PRIMARY, fontsize=12, fontweight="bold", pad=14,
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)

max_abs = max(max(brics_vals), max(ocde_vals), 1e-6)
ax.set_xlim(-max_abs * 1.2, max_abs * 1.2)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.0f}" for x in xticks], color=INK_MUTED, fontsize=9)

ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 8.3. Mapa combinado — Apply AI Strategy junto ao China New Gen, AI+ e PBIA entre BRICS e OCDE

Eixo X: similaridade com o **BRICS Declaration**. Eixo Y: similaridade com a
**OECD Recommendation**. Este mapa estende o da Seção 6.3, agora posicionando
os **quatro documentos nacionais/de bloco ao mesmo tempo** — PBIA (Brasil),
AI+ (China, 2023), China New Gen (China, 2017) e Apply AI Strategy (UE,
2025) — no mesmo eixo BRICS × OCDE. Como antes, cada documento aparece com
dois pontos (cosseno, o principal, e Jaccard, o secundário, ligados por uma
linha pontilhada), e os cantos de referência marcam onde o próprio BRICS e a
própria OCDE cairiam neste mapa.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

axis_max = 1.05
ax.plot([0, axis_max], [0, axis_max], linestyle="--", linewidth=1.2, color=INK_MUTED, zorder=2)

# Pontos de referência: onde o próprio BRICS e a própria OCDE cairiam neste mapa
ax.scatter([1.0], [cosine_similarity], s=140, color=CATEGORICAL_BLUE, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{BRICS_LABEL} (referência)")
ax.annotate(BRICS_LABEL, (1.0, cosine_similarity), textcoords="offset points", xytext=(-8, 8),
            ha="right", fontsize=8.5, color=INK_MUTED)

ax.scatter([cosine_similarity], [1.0], s=140, color=CATEGORICAL_AQUA, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{OECD_LABEL} (referência)")
ax.annotate(OECD_LABEL, (cosine_similarity, 1.0), textcoords="offset points", xytext=(8, -10),
            ha="left", fontsize=8.5, color=INK_MUTED)

# PBIA — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_pbia_brics, cosine_pbia_brics], [jaccard_pbia_ocde, cosine_pbia_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_PBIA, zorder=4)
ax.scatter([jaccard_pbia_brics], [jaccard_pbia_ocde], s=110, color=CATEGORICAL_PBIA, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="PBIA (Jaccard)")
ax.scatter([cosine_pbia_brics], [cosine_pbia_ocde], s=340, color=CATEGORICAL_PBIA, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="PBIA (cosseno)")
ax.annotate("PBIA", (cosine_pbia_brics, cosine_pbia_ocde), textcoords="offset points", xytext=(10, 10),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

# AI+ — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_aiplus_brics, cosine_aiplus_brics], [jaccard_aiplus_ocde, cosine_aiplus_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_AIPLUS, zorder=4)
ax.scatter([jaccard_aiplus_brics], [jaccard_aiplus_ocde], s=110, color=CATEGORICAL_AIPLUS, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="AI+ (Jaccard)")
ax.scatter([cosine_aiplus_brics], [cosine_aiplus_ocde], s=340, color=CATEGORICAL_AIPLUS, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="AI+ (cosseno)")
ax.annotate("AI+", (cosine_aiplus_brics, cosine_aiplus_ocde), textcoords="offset points", xytext=(10, -14),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

# China New Gen — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_china_brics, cosine_china_brics], [jaccard_china_ocde, cosine_china_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_CHINA, zorder=4)
ax.scatter([jaccard_china_brics], [jaccard_china_ocde], s=110, color=CATEGORICAL_CHINA, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="China New Gen (Jaccard)")
ax.scatter([cosine_china_brics], [cosine_china_ocde], s=340, color=CATEGORICAL_CHINA, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="China New Gen (cosseno)")
ax.annotate("China New Gen", (cosine_china_brics, cosine_china_ocde), textcoords="offset points", xytext=(10, -14),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

# Apply AI Strategy — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_apply_brics, cosine_apply_brics], [jaccard_apply_ocde, cosine_apply_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_APPLY, zorder=4)
ax.scatter([jaccard_apply_brics], [jaccard_apply_ocde], s=110, color=CATEGORICAL_APPLY, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="Apply AI Strategy (Jaccard)")
ax.scatter([cosine_apply_brics], [cosine_apply_ocde], s=340, color=CATEGORICAL_APPLY, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="Apply AI Strategy (cosseno)")
ax.annotate("Apply AI Strategy", (cosine_apply_brics, cosine_apply_ocde), textcoords="offset points", xytext=(10, 10),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

ax.set_xlabel(f"Similaridade com o {BRICS_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_ylabel(f"Similaridade com a {OECD_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_title("PBIA, AI+, China New Gen e Apply AI Strategy no mapa BRICS × OCDE",
             color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)
ax.set_xlim(0, axis_max)
ax.set_ylim(0, axis_max)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5)

fig.tight_layout()
plt.show()


### Síntese — Apply AI Strategy, BRICS, OCDE, PBIA, AI+ e China New Gen

- Contra a hipótese natural de que uma estratégia da Comissão Europeia
  penderia para o eixo de governança da OCDE, o resultado é o **oposto**: a
  **OCDE é o documento com que o Apply AI Strategy menos se parece**
  (cosseno 0,375) — abaixo até da similaridade com o AI+ chinês (0,398). O
  Apply AI Strategy se aproxima mais, em ordem, do **PBIA** (0,487), do
  **BRICS** (0,443), do **China New Gen** (0,429) e só depois do AI+ (0,398)
  e da OCDE (0,375, o menor de todos).
- O padrão se repete de forma notável: **os quatro documentos "nacionais"
  analisados — PBIA, AI+, China New Gen e Apply AI Strategy — penderam,
  todos, para o eixo do BRICS em vez do eixo da OCDE**, com margens de
  0,143 (PBIA), 0,087 (AI+), 0,068 (China New Gen) e 0,068 (Apply AI
  Strategy, empatado com o China New Gen). Nenhum dos quatro planos
  nacionais/regionais de IA se aproxima mais da OCDE do que do BRICS.
- Uma explicação plausível: PBIA, AI+, China New Gen e Apply AI Strategy são
  todos **planos estratégicos multissetoriais** organizados por setor, com
  ações, metas e investimentos concretos — compartilhando vocabulário
  estrutural (*strategy*, *sectors*, *support*, *development*, *innovation*)
  que os aproxima entre si e do BRICS (também rico em vocabulário de
  desenvolvimento e cooperação tecnológica). A OCDE, por sua vez, é uma
  recomendação de princípios de governança (*actors*, *stakeholders*,
  *lifecycle*, *trustworthy*), um registro linguístico distinto de qualquer
  um dos quatro planos nacionais — inclusive do europeu.

## 9. Síntese final — os seis documentos juntos

Estendemos o mapa de escalonamento multidimensional (MDS) da Seção 7 para
incluir o **Apply AI Strategy** como um sexto documento, ao lado de BRICS,
OCDE, AI+, PBIA e China New Gen. Construímos a matriz completa de distâncias
entre os seis documentos (distância = 1 − similaridade de cosseno) e
reaproveitamos a função `classical_mds` já definida na Seção 5.2, mantendo os
dois componentes de maior variância.

In [ ]:
doc_labels6 = [BRICS_LABEL, OECD_LABEL, AIPLUS_LABEL, PBIA_LABEL, CHINA_LABEL, APPLY_LABEL]
doc_colors6 = [CATEGORICAL_BLUE, CATEGORICAL_AQUA, CATEGORICAL_AIPLUS, CATEGORICAL_PBIA, CATEGORICAL_CHINA, CATEGORICAL_APPLY]

# Matriz de similaridade de cosseno (ordem: BRICS, OCDE, AI+, PBIA, China New Gen, Apply AI Strategy)
S6 = np.array([
    [1.0,                 cosine_similarity,    cosine_aiplus_brics, cosine_pbia_brics, cosine_china_brics, cosine_apply_brics],
    [cosine_similarity,   1.0,                  cosine_aiplus_ocde,  cosine_pbia_ocde,  cosine_china_ocde,  cosine_apply_ocde],
    [cosine_aiplus_brics, cosine_aiplus_ocde,   1.0,                 cosine_aiplus_pbia, cosine_china_aiplus, cosine_apply_aiplus],
    [cosine_pbia_brics,   cosine_pbia_ocde,     cosine_aiplus_pbia,  1.0,                cosine_china_pbia,  cosine_apply_pbia],
    [cosine_china_brics,  cosine_china_ocde,    cosine_china_aiplus, cosine_china_pbia,  1.0,                cosine_apply_china],
    [cosine_apply_brics,  cosine_apply_ocde,    cosine_apply_aiplus, cosine_apply_pbia,  cosine_apply_china, 1.0],
])
D6 = 1.0 - S6
np.fill_diagonal(D6, 0.0)

coords6, explained_variance6 = classical_mds(D6)
print("Coordenadas MDS (dim 1, dim 2):")
for label, (x, y) in zip(doc_labels6, coords6):
    print(f"  {label:24s} ({x:+.3f}, {y:+.3f})")
print(f"\nVariância explicada pelos 2 componentes: {explained_variance6*100:.1f}%")

fig, ax = plt.subplots(figsize=(8.5, 8.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

# Linhas conectando todos os pares, com espessura proporcional à similaridade
# (par mais similar = linha mais grossa e mais opaca)
for i in range(len(doc_labels6)):
    for j in range(i + 1, len(doc_labels6)):
        sim = S6[i, j]
        ax.plot([coords6[i, 0], coords6[j, 0]], [coords6[i, 1], coords6[j, 1]],
                color=INK_MUTED, linewidth=0.6 + sim * 3.2, alpha=0.18 + sim * 0.35, zorder=1)

for label, color, (x, y) in zip(doc_labels6, doc_colors6, coords6):
    ax.scatter([x], [y], s=420, color=color, alpha=0.9, edgecolors=SURFACE,
               linewidths=1.6, zorder=3)
    ax.annotate(label, (x, y), textcoords="offset points", xytext=(0, 20),
                ha="center", fontsize=11, fontweight="bold", color=INK_PRIMARY, zorder=4)

ax.axhline(0, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.axvline(0, color=GRIDLINE, linewidth=0.8, zorder=0)

pad = max(np.abs(coords6).max() * 0.45, 0.05)
ax.set_xlim(coords6[:, 0].min() - pad, coords6[:, 0].max() + pad)
ax.set_ylim(coords6[:, 1].min() - pad, coords6[:, 1].max() + pad)

ax.set_xlabel("Dimensão MDS 1", color=INK_MUTED, fontsize=10)
ax.set_ylabel("Dimensão MDS 2", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Mapa de similaridade textual dos seis documentos (MDS clássico, {explained_variance6*100:.0f}% da variância)",
    color=INK_PRIMARY, fontsize=12.5, fontweight="bold", pad=16,
)

for spine in ax.spines.values():
    spine.set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()


### Síntese — o mapa completo dos seis documentos

- A variância explicada pelos dois componentes cai para **66,5%** (ante
  80,3% com cinco documentos) — com seis documentos, o espaço de
  similaridade textual já não cabe tão bem em duas dimensões, então o mapa
  deve ser lido como um retrato aproximado da estrutura geral, não como
  distâncias exatas par a par.
- A ligação mais forte de todo o conjunto de seis documentos permanece
  **China New Gen–AI+ (0,742)**, seguida por **BRICS–PBIA (0,538)** e
  **BRICS–OCDE (0,519)** — a hierarquia observada nas seções anteriores não
  muda com a entrada do Apply AI Strategy.
- O achado mais importante desta seção é que a **OCDE se confirma como o
  documento mais isolado de todo o estudo**: suas quatro menores
  similaridades de todo o conjunto de 15 pares são, justamente, as que ela
  tem com os quatro planos nacionais/regionais — PBIA (0,395), China New Gen
  (0,379), Apply AI Strategy (0,375) e AI+ (0,330, a mais baixa de todas). A
  única conexão relevante da OCDE é com o **BRICS** (0,519) — as duas únicas
  "declarações de princípios" do conjunto, em contraste com os quatro planos
  de ação setoriais. No mapa, a OCDE aparece isolada no canto inferior
  esquerdo, afastada de todos os demais pontos.
- O Apply AI Strategy, por sua vez, aparece próximo do **PBIA** no mapa
  (ambos no quadrante superior) — reforçando que a proximidade entre os dois
  vem da estrutura setorial compartilhada dos planos, não de afinidade
  política de bloco. Já **AI+** e **China New Gen** seguem formando o par
  visualmente mais próximo de toda a figura, à direita, refletindo a mesma
  autoria por trás dos dois documentos.
- Em conjunto, o mapa de seis documentos sugere três agrupamentos: (1) as
  duas políticas chinesas (AI+ e China New Gen), (2) os planos setoriais de
  PBIA e Apply AI Strategy, com o BRICS funcionando como elo próximo de
  ambos, e (3) a OCDE, isolada como o único documento de governança pura do
  conjunto.